# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/row-huh/ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Same working set as w02-w04: 30-day-plus-old, impression-positive rows, `avg_position==0` ("no data") dropped → **28,795 rows**. Numeric features get median imputation where missing, plus an explicit `has_word_count` flag rather than a silent 0-fill (`word_count` is missing on 26.7% of rows and, per the data dictionary, missingness tracks `content_type` — filling blind would inject a fake category signal). Categoricals get an explicit `"missing"` bucket instead of being dropped, so a null `main_intent`/`competition_level` is still usable information, not a lost row.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
qualified = df[(df.impressions_90d > 0) & (df.content_age_days >= 90)].copy()
q = qualified[qualified.avg_position > 0].copy().reset_index(drop=True)

q["has_word_count"] = q.word_count.notna().astype(int)

num_feats = ["avg_position", "impressions_90d", "word_count", "content_age_days",
             "days_since_last_update", "scroll_rate", "engagement_rate", "has_word_count"]
cat_feats = ["content_type", "main_intent", "competition_level", "freshness_tier"]

feature_vector = q[num_feats + cat_feats].copy()
print("feature vector shape:", feature_vector.shape)
feature_vector.head(3)


feature vector shape: (28795, 12)


,avg_position,impressions_90d,word_count,content_age_days,days_since_last_update,scroll_rate,engagement_rate,has_word_count,content_type,main_intent,competition_level,freshness_tier
0,10.6,3803,3221.0,187,20,4.55,5.88,1,keyword article,transactional,HIGH,0-30
1,20.3,15320,2481.0,445,25,10.00,0.00,1,keyword article,informational,LOW,0-30
2,36.5,12581,3515.0,141,20,28.57,0.00,1,keyword article,informational,LOW,0-30


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available before review? |
|---|---|---|---|
| `avg_position` | current avg search rank | n/a (0="no data" already dropped) | yes — current state |
| `impressions_90d` | trailing 90-day impressions | n/a (filter requires >0) | yes |
| `word_count` | content length | median-impute + `has_word_count` flag (26.7% missing) | yes |
| `content_age_days` | days since publish | n/a, none missing | yes |
| `days_since_last_update` | staleness | n/a, none missing | yes |
| `scroll_rate` | % scroll depth signal | median-impute (0.4% missing) | yes |
| `engagement_rate` | engagement signal | n/a, none missing | yes |
| `content_type` / `main_intent` / `competition_level` / `freshness_tier` | categorical buckets | `"missing"` category (main_intent 5.6%, competition_level 6.5% missing) | yes |

All eight are current-state, current-window fields — every one exists at the moment a reviewer would look at the page, so none fail the *temporal* availability test. That does **not** mean none are risky — the leakage hunt below is a separate question about whether a feature is too close to the *label itself*, not whether it's from the future.

In [2]:
print("missing rate per feature:")
print(feature_vector.isna().mean().round(3))


missing rate per feature:
avg_position              0.000
impressions_90d           0.000
word_count                0.267
content_age_days          0.000
days_since_last_update    0.000
scroll_rate               0.004
engagement_rate           0.000
has_word_count            0.000
content_type              0.000
main_intent               0.056
competition_level         0.065
freshness_tier            0.000
dtype: float64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Label (same as w04/w05): `impressions_90d>=500 & avg_position<=20 & ctr<0.5`.

**Single-feature attack (AUC of each raw feature vs. the label):** nothing crosses the near-perfect 0.95 line alone — `ctr` itself, the field most directly defining the label, only scores 0.636 solo, because it's one of three ANDed conditions and doesn't separate the label by itself. `impressions_90d` is the highest at 0.823 ("watch", not "suspicious").

**But single-feature checks aren't enough.** w05 found Random Forest hit precision@20 ≈1.0 combining `avg_position` + `impressions_90d` — two features that individually looked fine. Reproducing that here with a plain *linear* combination of just those two gets AUC 0.740, barely above either alone — a linear check misses it. The leakage only shows up once a nonlinear model is allowed to build an AND-like split across both, because together they reconstruct 2 of the label's 3 defining conditions. **Lesson: leakage can hide inside model structure even when no single feature, or simple linear combination, looks alarming.**

**Other checks:** the 30-day sub-windows never exceed the 90-day total (0.0% violation — consistent, not overlapping oddly), and `trend_direction`/`trend_pct` are excluded on principle regardless of this label, since the data dictionary derives one from the other (the project's universal "label trap").

In [3]:
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

label = ((q.impressions_90d >= 500) & (q.avg_position <= 20) & (q.ctr < 0.5)).astype(int)

attack_fields = ["ctr", "avg_position", "impressions_90d", "clicks_90d", "word_count",
                  "content_age_days", "days_since_last_update", "scroll_rate", "engagement_rate",
                  "trend_pct", "impressions_last_30d", "impressions_prev_30d",
                  "ai_traffic_pct", "ai_sessions_90d"]

print("single-feature AUC vs label:")
for c in attack_fields:
    vals = q[c]
    mask = vals.notna()
    auc = roc_auc_score(label[mask], vals[mask])
    auc = max(auc, 1 - auc)
    flag = " <-- SUSPICIOUS" if auc > 0.95 else (" <-- watch" if auc > 0.75 else "")
    print(f"{c:26s} AUC={auc:.3f}{flag}")

# combined linear check: does a simple linear model on just the two "watch" fields jump to suspicious?
X = StandardScaler().fit_transform(q[["avg_position", "impressions_90d"]].values)
combined_auc = roc_auc_score(label, LogisticRegression(max_iter=1000).fit(X, label).predict_proba(X)[:, 1])
print("\ncombined LINEAR AUC (avg_position + impressions_90d):", round(combined_auc, 3),
      "-- w05's nonlinear (Random Forest) model on the same two fields hit precision@20~1.0")

# window overlap consistency
sum30 = q.impressions_last_30d.fillna(0) + q.impressions_prev_30d.fillna(0)
print("\npct rows where 30d sub-windows exceed the 90d total:", round((sum30 > q.impressions_90d).mean() * 100, 2))


single-feature AUC vs label:
ctr                        AUC=0.636
avg_position               AUC=0.699
impressions_90d            AUC=0.823 <-- watch
clicks_90d                 AUC=0.745
word_count                 AUC=0.518
content_age_days           AUC=0.512
days_since_last_update     AUC=0.531
scroll_rate                AUC=0.524
engagement_rate            AUC=0.590
trend_pct                  AUC=0.518
impressions_last_30d       AUC=0.796 <-- watch
impressions_prev_30d       AUC=0.807 <-- watch
ai_traffic_pct             AUC=0.502
ai_sessions_90d            AUC=0.502

combined LINEAR AUC (avg_position + impressions_90d): 0.74 -- w05's nonlinear (Random Forest) model on the same two fields hit precision@20~1.0

pct rows where 30d sub-windows exceed the 90d total: 0.0


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **`ctr`** — literally one of the label's three ANDed conditions; definitional leakage, not just correlation.
- **`trend_direction`, `trend_pct`** — one is derived from the other (the data dictionary's own label-trap warning); excluded on principle for every label in this project, not just this one.
- **`content_id`, `client_id`** — pseudonymous join/grouping keys only, used for the client-grouped split, never as feature values.
- **`char_count`** — 0.94 correlated with `word_count` (from w03); redundant, kept the less-collinear one.
- **`provider_used`, `model_used`** — describe how content was produced, not a search/engagement signal, and mostly missing (71.5% / 19.1% null, from w03).
- **`ai_sessions_90d`** — AUC 0.502, essentially noise for this label, and only 6.4% of rows have any AI session at all.
- **`impressions_last_30d`, `impressions_prev_30d`** — each AUC ~0.8, largely redundant with `impressions_90d` (they sum to it, confirmed above); adding them wouldn't add real new information, just more surface for the same combined-feature risk flagged in section 3.
- **`avg_position`, `impressions_90d`** — *kept* in w05's "full" comparison to represent realistic available signal, but flagged there and reconfirmed here as the actual source of the suspiciously-high result; a stricter, leakage-free feature set drops these too (that's exactly what w05's "harder" model did, precision@50 falling from 0.98 to a more honest 0.48).

In [4]:
excluded = {
    "ctr": "literal label component (definitional leakage)",
    "trend_direction": "derived from trend_pct (label-trap rule, project-wide)",
    "trend_pct": "source of trend_direction (label-trap rule, project-wide)",
    "content_id": "pseudonym, grouping key only",
    "client_id": "pseudonym, grouping key only",
    "char_count": "0.94 correlated with word_count, redundant",
    "provider_used": "describes production, not a search signal; mostly missing",
    "model_used": "describes production, not a search signal; mostly missing",
    "ai_sessions_90d": "AUC 0.502, noise-level for this label; 93.6% zero",
    "impressions_last_30d": "redundant sub-window of impressions_90d",
    "impressions_prev_30d": "redundant sub-window of impressions_90d",
}
for field, reason in excluded.items():
    print(f"{field:24s} {reason}")


ctr                      literal label component (definitional leakage)
trend_direction          derived from trend_pct (label-trap rule, project-wide)
trend_pct                source of trend_direction (label-trap rule, project-wide)
content_id               pseudonym, grouping key only
client_id                pseudonym, grouping key only
char_count               0.94 correlated with word_count, redundant
provider_used            describes production, not a search signal; mostly missing
model_used               describes production, not a search signal; mostly missing
ai_sessions_90d          AUC 0.502, noise-level for this label; 93.6% zero
impressions_last_30d     redundant sub-window of impressions_90d
impressions_prev_30d     redundant sub-window of impressions_90d


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.